## Parallel Workflows ##

**We always update partial state in paralle workflows where evry node or function return only the updated field or changed field existing are passes as it is.**

In [50]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from pydantic import json


# Define State Schema
class BatsMan(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    bpb: float
    boundary_percent: float
    summary: str


# Node Functions (Each returning partial state updates)
def calculate_sr(state: BatsMan) -> dict:
    sr = (state["runs"] / state["balls"]) * 100 if state["balls"] > 0 else 0.0
    return {"sr": round(sr, 2)}


def calculate_bpb(state: BatsMan) -> dict:
    total_boundaries = state["fours"] + state["sixes"]
    bpb = state["balls"] / total_boundaries if total_boundaries > 0 else 0.0
    return {"bpb": round(bpb, 2)}


def calculate_boundary_percent(state: BatsMan) -> dict:
    boundary_runs = (state["fours"] * 4) + (state["sixes"] * 6)
    boundary_percent = (boundary_runs / state["runs"]) * 100 if state["runs"] > 0 else 0.0
    return {"boundary_percent": round(boundary_percent, 2)}


def summary(state: BatsMan) -> dict:
    summary_text = f"""
Strike Rate - {state['sr']}
Balls per boundary - {state['bpb']}
Boundary percent - {state['boundary_percent']}%
"""
    return {"summary": summary_text}


# Build Graph
graph = StateGraph(BatsMan)

# Add Nodes
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)

# Add Edges (Fan-out from START to calculations, Fan-in to summary)
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percent", "summary")

graph.add_edge("summary", END)

# Compile Workflow
workflow = graph.compile()

# Execute Workflow
initial_state = {
    "runs": 85,
    "balls": 42,
    "fours": 8,
    "sixes": 4
}

final_state = workflow.invoke(initial_state)

# Display Output
print("Final State Dictionary:")
print(final_state)

print("\nFormatted Summary:")
print(final_state["summary"])

Final State Dictionary:
{'runs': 85, 'balls': 42, 'fours': 8, 'sixes': 4, 'sr': 202.38, 'bpb': 3.5, 'boundary_percent': 65.88, 'summary': '\nStrike Rate - 202.38\nBalls per boundary - 3.5\nBoundary percent - 65.88%\n'}

Formatted Summary:

Strike Rate - 202.38
Balls per boundary - 3.5
Boundary percent - 65.88%



## LLM Parallel Workflow ##

In [3]:
import os
import operator
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

# ================================
# 1. SCHEMAS & STATE DEFINITION
# ================================

# Pydantic schema for structured output so dot-notation (output.feedback, output.score) works
class AspectEvaluation(BaseModel):
    feedback: str = Field(description="Detailed evaluation feedback for the evaluated aspect")
    score: int = Field(description="Score out of 10")

# LangGraph State Schema
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_feedback: Annotated[list[int], operator.add]
    avg_score: float

# ================================
# 2. MODEL INITIALIZATION
# ================================
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
structured_output = model.with_structured_output(AspectEvaluation)

# ================================
# 3. NODE FUNCTIONS
# ================================
def language_feedback(state: EssayState):
    prompt = f"You are an expert English Literature evaluator. Evaluate the language and grammar of the following essay:\n\n{state['essay']}"
    output = structured_output.invoke(prompt)

    return {
        "language_feedback": output.feedback,
        "individual_feedback": [output.score]
    }

def analysis_feedback(state: EssayState):
    prompt = f"You are an expert essay evaluator. Evaluate the depth of analysis and argument quality of the following essay:\n\n{state['essay']}"
    output = structured_output.invoke(prompt)

    return {
        "analysis_feedback": output.feedback,
        "individual_feedback": [output.score]
    }

def clarity_feedback(state: EssayState):
    prompt = f"You are an expert essay evaluator. Evaluate the clarity of thought and logical flow of the following essay:\n\n{state['essay']}"
    output = structured_output.invoke(prompt)

    return {
        "clarity_feedback": output.feedback,
        "individual_feedback": [output.score]
    }

def overall_feedback(state: EssayState):
    # Summary feedback prompt
    prompt = (
        f"Based on the following individual feedbacks, create a summarized overall feedback:\n"
        f"Language feedback: {state.get('language_feedback', 'N/A')}\n"
        f"Analysis feedback: {state.get('analysis_feedback', 'N/A')}\n"
        f"Clarity feedback: {state.get('clarity_feedback', 'N/A')}"
    )
    summary_response = model.invoke(prompt).content

    # Calculate average score
    scores = state.get("individual_feedback", [])
    avg_score = sum(scores) / len(scores) if scores else 0.0

    return {
        "overall_feedback": summary_response,
        "avg_score": avg_score
    }

# ================================
# 4. BUILD GRAPH & EDGES
# ================================
graph = StateGraph(EssayState)

# Add Nodes
graph.add_node("language_feedback", language_feedback)
graph.add_node("analysis_feedback", analysis_feedback)
graph.add_node("clarity_feedback", clarity_feedback)
graph.add_node("overall_feedback", overall_feedback)

# Add Edges (Parallel evaluation nodes -> Consolidated summary)
graph.add_edge(START, "language_feedback")
graph.add_edge(START, "analysis_feedback")
graph.add_edge(START, "clarity_feedback")

graph.add_edge("language_feedback", "overall_feedback")
graph.add_edge("analysis_feedback", "overall_feedback")
graph.add_edge("clarity_feedback", "overall_feedback")

graph.add_edge("overall_feedback", END)

# ================================
# 5. COMPILE & EXECUTE
# ================================
app = graph.compile()

sample_essay = """
Artificial Intelligence is rapidly reshaping modern society. On one hand, automation increases efficiency
and streamlines repetitive tasks across industries. On the other hand, rapid deployment raises ethical concerns
and potential job displacement. To ensure sustainable growth, policies must balance innovation with social welfare.
"""

initial_state = {"essay": sample_essay}
result = app.invoke(initial_state)

print("=== EVALUATION RESULT ===")
print("Individual Scores:", result["individual_feedback"])
print("Average Score:", result["avg_score"])
print("\nLanguage Feedback:", result["language_feedback"])
print("\nAnalysis Feedback:", result["analysis_feedback"])
print("\nClarity Feedback:", result["clarity_feedback"])
print("\nOverall Summary Feedback:\n", result["overall_feedback"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 38.716563268s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url